# FastVLA: Ultra-Efficient VLA Fine-Tuning on Tesla T4 (Distributed)

FastVLA is designed to democratize Vision-Language-Action (VLA) models. This notebook supports **distributed training on Kaggle's 2x T4 GPUs** with automatic mixed precision and gradient accumulation.

### Why this matters?
- Standard OpenVLA-7B (FP16) requires ~28GB VRAM (impossible on T4).
- FastVLA uses 4-bit QLoRA and Custom Triton Kernels to reduce memory by 70%.
- **Distributed Training**: Automatically utilizes both T4 GPUs on Kaggle.
- You get 6x faster iterations than standard implementations.

> **Goal:** Fine-tune OpenVLA for 50 steps on the PushT robotics dataset in under 2 minutes.

## ⚠️ Important: Gated Model Access (Llama-2)

OpenVLA internally uses **Llama-2-7B** as its language backbone. Meta requires all users to manually request access to Llama-2. 

**If you haven't been approved yet:**
1. Visit [meta-llama/Llama-2-7b-hf](https://huggingface.co/meta-llama/Llama-2-7b-hf) and click **Request Access**.
2. **Wait for Approval**: This can take anywhere from 1 hour to 2 days.
3. **Login**: Once approved, use your HF token in the cell below.

**🚀 Want to skip the wait? (Instant Mode)**
If you want to start training **immediately** without waiting for Meta's approval, simply change the `model_id` in Step 2 to `"smolvla"`. It is non-gated and works instantly!

## 0. Authentication
1. **Add Token**: 
   - **Kaggle**: Go to **Add-ons -> Secrets**, add `HF_TOKEN` with your HuggingFace API key.
   - **Colab**: Click the **🔑 (Secrets)** icon, add `HF_TOKEN` and enable 'Notebook access'.

In [1]:
from huggingface_hub import login
import os

# Check for Kaggle Secrets or Colab Secrets
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    token = user_secrets.get_secret("HF_TOKEN")
    login(token=token)
except:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
        login(token=token)
    except:
        # Manual fallback
        print("No HF_TOKEN found in secrets. Redirecting to manual login...")
        login()

## 1. Setup Environment
We install FastVLA and its optimized dependencies. 

**Tip for Kaggle users:** You can attach the 'openvla' model directly from the '+ Add Model' tab in the right sidebar to bypass the 15GB download and start training instantly!

In [2]:
# ⚠️ CRITICAL: Install Unsloth FIRST (before transformers/peft)
# Unsloth must be imported before transformers to apply optimizations
!pip install -q --upgrade huggingface_hub --no-cache-dir
!pip install -q unsloth_zoo --no-cache-dir
!pip install -q git+https://github.com/unslothai/unsloth.git --no-cache-dir

# Install FastVLA from GitHub
!pip install -q git+https://github.com/BouajilaHamza/FastVLA.git --no-cache-dir

# Install other dependencies
!pip install -q triton bitsandbytes accelerate peft transformers datasets timm --no-cache-dir

# ⚠️ CRITICAL: Import Unsloth FIRST to apply patches
import unsloth
print('✓ Unsloth imported first (patches applied)')

# Now verify FastVLA installation
print('\n' + '='*60)
print('Environment Diagnostic')
print('='*60)

try:
    import fastvla
    print(f'✓ FastVLA imported from: {fastvla.__file__}')
    
    # Check if Unsloth detection works
    from fastvla import model as fastvla_model
    print(f'  UNSLOTH_AVAILABLE: {fastvla_model.UNSLOTH_AVAILABLE}')
    
    if not fastvla_model.UNSLOTH_AVAILABLE:
        print('  ⚠ Unsloth detection failed! Attempting patch...')
        # Try to import Unsloth components manually
        try:
            from unsloth import FastLanguageModel, FastVisionModel, patch_forward, patch_model, patch_saving_functions
            fastvla_model.UNSLOTH_AVAILABLE = True
            fastvla_model.FastLanguageModel = FastLanguageModel
            fastvla_model.FastVisionModel = FastVisionModel
            fastvla_model.patch_forward = patch_forward
            fastvla_model.patch_model = patch_model
            fastvla_model.patch_saving_functions = patch_saving_functions
            print('  ✓ Manually enabled Unsloth imports')
        except Exception as e:
            print(f'  ✗ Manual patch failed: {e}')
except Exception as e:
    print(f'✗ FastVLA import failed: {e}')

print('='*60)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 637.3/637.3 kB 13.9 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 130.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.5/417.5 kB 4.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 4.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 6.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 9.9 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 13.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.9/224.9 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 15.4 MB/s eta 0

ImportError: cannot import name 'BucketNotFoundError' from 'huggingface_hub.errors' (/usr/local/lib/python3.12/dist-packages/huggingface_hub/errors.py)

## 2. Load Model in 4-bit (QLoRA)
We load OpenVLA-7B with 4-bit quantization. 

**Important:** The `action_dim` parameter must match your dataset's action dimensions. For PushT, this is 2 (x, y coordinates). For other robotics datasets, it's typically 7 (x, y, z, roll, pitch, yaw, gripper).

**Instant Start Tip:** To skip Llama-2 gated access issues, change `model_id` below to `"smolvla"`.

In [3]:
!pip show unsloth

Name: unsloth
Version: 2026.4.4
Summary: 2-5X faster training, reinforcement learning & finetuning
Home-page: https://unsloth.ai
Author: Unsloth AI team
Author-email: info@unsloth.ai
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: nest-asyncio, pydantic, pyyaml, typer
Required-by: 


In [5]:
from fastvla import FastVLAModel
import torch

model_id = "openvla/openvla-7b" # Change to "smolvla" for non-gated, instant access

# IMPORTANT: action_dim must match your dataset. PushT uses 2D actions (x, y).
# Standard robotics datasets often use 7D (x, y, z, roll, pitch, yaw, gripper).
ACTION_DIM = 2  # Change this to match your dataset!

print(f"Loading {model_id} in 4-bit...")
model = FastVLAModel.from_pretrained(
    model_id,
    load_in_4bit=True,
    use_peft=True,
    gradient_checkpointing=True,
    action_dim=ACTION_DIM,  # Must match your dataset!
)

print(f"Model loaded. Current VRAM: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

Loading openvla/openvla-7b in 4-bit...


ImportError: Cannot load vision encoder in 4-bit: Unsloth is not installed.
4-bit QLoRA loading requires Unsloth for proper quantization.

Fix — choose one:
  1. Install Unsloth:
     pip install git+https://github.com/unslothai/unsloth.git
  2. Set load_in_4bit=False (model loads in FP32, higher VRAM)
     e.g. FastVLAModel.from_pretrained(..., load_in_4bit=False)

## 3. Fine-Tuning on PushT (Real Robotics)
Next, we load the lerobot/pusht_image dataset and run an optimized training loop with distributed training support.

**Distributed Training Features:**
- Automatically uses both T4 GPUs on Kaggle
- Mixed precision (fp16) for faster training
- 8-bit optimizer for memory efficiency
- Gradient accumulation for larger effective batch sizes

In [4]:
from fastvla import FastVLATrainer, get_dataset

# Load only a small subset for demonstration
dataset = get_dataset("lerobot/pusht_image")

# Verify action dimensions match
sample_action = dataset[0]["actions"]
print(f"Dataset action shape: {sample_action.shape}")
print(f"Model action_dim: {model.config.action_dim}")
assert sample_action.shape[-1] == model.config.action_dim, \
    f"Action dimension mismatch! Dataset: {sample_action.shape[-1]}, Model: {model.config.action_dim}"

trainer = FastVLATrainer(
    model=model,
    dataset=dataset,
    batch_size=2,  # Per-GPU batch size
    lr=1e-4,
    max_steps=50,  # Set to 2000 for full convergence
    # Distributed training optimizations
    use_mixed_precision=True,  # fp16 for T4 GPUs
    use_8bit_optimizer=True,   # Memory efficient
    gradient_accumulation_steps=2,  # Effective batch size = 4
    # Checkpointing and logging
    output_dir="./checkpoints",
    save_steps=50,
    logging_steps=10,
)

print("Starting Distributed Training...")
results = trainer.train()

print(f"Training Complete! Final loss: {results[-1]['loss']:.4f}")

📥 Loading dataset lerobot/pusht_image from HuggingFace...
Dataset action shape: torch.Size([2])
Model action_dim: 2
Starting Distributed Training...


Epoch 1/1:   0%|          | 0/500 [00:00<?, ?it/s]/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
Epoch 1/1:   0%|          | 0/500 [00:02<?, ?it/s]


RuntimeError: expected m1 and m2 to have the same dtype, but got: c10::Half != float

## 4. Inference (Predict Action)
Now we test the model by giving it an image and a text prompt to predict the robot's next action tensor.

In [ ]:
import numpy as np
from PIL import Image

# Dummy image for demonstration
dummy_image = Image.fromarray(np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8))
prompt = "push the t-shaped block into the goal zone"

# Process the image and prompt through the model
from torchvision import transforms
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
])

pixel_values = transform(dummy_image).unsqueeze(0).unsqueeze(0)  # [1, 1, C, H, W]
input_ids = model.tokenizer(prompt, return_tensors="pt")["input_ids"]

with torch.no_grad():
    action, _ = model(pixel_values=pixel_values, input_ids=input_ids)

print(f"Predicted Action Tensor ({model.config.action_dim}D):")
print(action)

---
**Star the repo** to support democratized robotics!
[GitHub: FastVLA](https://github.com/BouajilaHamza/FastVLA)

**Created by the FastVLA Team.**